# Étape 5 — Manipulations : explorer une ontologie RDFS

On charge l'ontologie des films et on l'explore : d'abord le **schéma** (classes, sous-classes, propriétés), puis les **individus**. Pas encore d'inférence ici — on observe seulement ce qui est écrit explicitement.

> Prérequis : `pip install -r requirements.txt`. Exécute les cellules dans l'ordre.

In [ ]:
from rdflib import Graph

graphe = Graph()
graphe.parse("films.ttl", format="turtle")
print(f"{len(graphe)} triplets charges.\n")


def afficher(requete: str) -> None:
    resultats = graphe.query(requete)
    if not resultats:
        print("  (aucun resultat)")
        return
    for ligne in resultats:
        valeurs = {str(var): str(val) for var, val in ligne.asdict().items()}
        print(" ", valeurs)

41 triplets charges.



## 1. Quelles classes existent ?

On interroge le **schéma** : tout ce qui est déclaré comme `rdfs:Class`.

In [ ]:
afficher("""
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?classe ?label
WHERE {
    ?classe a rdfs:Class .
    ?classe rdfs:label ?label .
}
""")

  {'classe': 'http://exemple.org/cinema/Personne', 'label': 'Personne'}
  {'classe': 'http://exemple.org/cinema/Film', 'label': 'Film'}
  {'classe': 'http://exemple.org/cinema/Realisateur', 'label': 'Realisateur'}
  {'classe': 'http://exemple.org/cinema/Acteur', 'label': 'Acteur'}


## 2. Quelles sont les sous-classes de Personne ?

On lit la hiérarchie via `rdfs:subClassOf`.

In [ ]:
afficher("""
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX ex:   <http://exemple.org/cinema/>

SELECT ?sousClasse
WHERE {
    ?sousClasse rdfs:subClassOf ex:Personne .
}
""")

  {'sousClasse': 'http://exemple.org/cinema/Realisateur'}
  {'sousClasse': 'http://exemple.org/cinema/Acteur'}


## 3. Les propriétés et leur domain / range

On lit la « signature » de chaque propriété : quel type de sujet (`domain`) et quel type d'objet (`range`) elle attend.

In [ ]:
afficher("""
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?propriete ?domain ?range
WHERE {
    ?propriete a rdf:Property .
    ?propriete rdfs:domain ?domain .
    ?propriete rdfs:range ?range .
}
""")

  {'propriete': 'http://exemple.org/cinema/aRealise', 'domain': 'http://exemple.org/cinema/Realisateur', 'range': 'http://exemple.org/cinema/Film'}
  {'propriete': 'http://exemple.org/cinema/aJoueDans', 'domain': 'http://exemple.org/cinema/Acteur', 'range': 'http://exemple.org/cinema/Film'}
  {'propriete': 'http://exemple.org/cinema/titre', 'domain': 'http://exemple.org/cinema/Film', 'range': 'http://www.w3.org/2001/XMLSchema#string'}
  {'propriete': 'http://exemple.org/cinema/nom', 'domain': 'http://exemple.org/cinema/Personne', 'range': 'http://www.w3.org/2001/XMLSchema#string'}


## 4. Le piège à comprendre avant l'étape 6

Demandons **toutes les Personnes**. Intuitivement, on s'attend à voir Nolan, DiCaprio, Eastwood...

Mais SANS raisonneur, le résultat est **vide** : aucun individu n'est écrit explicitement comme `ex:Personne` (ils sont typés `Realisateur` ou `Acteur`). RDFS « sait » que ce sont des sous-classes de Personne, mais tant qu'on n'active pas l'inférence, cette connaissance n'est pas matérialisée.

In [ ]:
# Resultat attendu : VIDE (aucun individu n'est explicitement de type Personne)
afficher("""
PREFIX ex: <http://exemple.org/cinema/>

SELECT ?personne
WHERE {
    ?personne a ex:Personne .
}
""")

  (aucun resultat)


En attendant l'étape 6, on peut malgré tout retrouver les personnes « à la main » en demandant explicitement les Réalisateurs **et** les Acteurs avec un `UNION` :

In [ ]:
afficher("""
PREFIX ex: <http://exemple.org/cinema/>

SELECT DISTINCT ?nom
WHERE {
    { ?p a ex:Realisateur } UNION { ?p a ex:Acteur }
    ?p ex:nom ?nom .
}
""")

  {'nom': 'Christopher Nolan'}
  {'nom': 'Clint Eastwood'}
  {'nom': 'Leonardo DiCaprio'}


Cette requête `UNION` est lourde et fragile : si on ajoute demain une classe `Scenariste`, il faudra penser à la rajouter ici. **L'inférence (étape 6) va rendre ce bricolage inutile** : on pourra simplement demander « les Personnes ».

## À toi de jouer

**Exercice :** écris une requête qui liste **les films et leur titre**, ainsi que le **nom de leur réalisateur**.

Indice : relie `?realisateur ex:aRealise ?film`, puis va chercher `ex:nom` et `ex:titre`.

In [ ]:
# TODO : complete cette requete
afficher("""
PREFIX ex: <http://exemple.org/cinema/>

SELECT ?titre ?nomRealisateur
WHERE {
    ?realisateur ex:aRealise ?film .
    # ... a completer ...
}
""")

<details>
<summary>Afficher la solution</summary>

```python
afficher("""
PREFIX ex: <http://exemple.org/cinema/>

SELECT ?titre ?nomRealisateur
WHERE {
    ?realisateur ex:aRealise ?film .
    ?realisateur ex:nom ?nomRealisateur .
    ?film ex:titre ?titre .
}
""")
```

Résultat : « Inception » → Christopher Nolan, « Gran Torino » → Clint Eastwood.

</details>

---

Tu sais décrire un schéma. À l'**étape 6**, on branche le raisonneur et on regarde la magie de l'inférence opérer.